<a href="https://colab.research.google.com/github/melissa-04/tubitak-2209a-spatial-stemness-emt/blob/main/07_spatial_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scanpy statsmodels "pandas==2.3.3"

from google.colab import drive
drive.mount('/content/drive')

import os, sys
import pandas as pd, numpy as np
import scipy
from scipy.stats import spearmanr, wilcoxon, mannwhitneyu
import statsmodels

BASE    = '/content/drive/MyDrive/Tubitak-2209a'
RESULTS = f'{BASE}/04_results'
FIGURES = f'{BASE}/05_figures'
os.makedirs(RESULTS, exist_ok=True)

print(f'python {sys.version.split()[0]} | pandas {pd.__version__} | '
      f'numpy {np.__version__} | scipy {scipy.__version__} | statsmodels {statsmodels.__version__}')

scores = pd.read_csv(f'{BASE}/03_spatial_analysis/spatial_all_scores.csv')
regions = pd.read_csv(f'{BASE}/01_processed_data/combined_with_regions.csv')
regions['SpotID'] = regions.patientid.astype(str) + '_' + regions.barcode.astype(str)

df = scores.merge(
    regions[['SpotID', 'array_row', 'array_col', 'nCount_RNA', 'nFeature_RNA']],
    on='SpotID', how='left')

print(f'\nMerged table: {df.shape[0]} spots x {df.shape[1]} columns')
print(f'  coordinates present : {df.array_row.notna().sum()}')
print(f'  stemness defined    : {df.spatial_stemness.notna().sum()}')

PATIENTS = sorted(df.patientid.unique())
SUBTYPE  = df.groupby('patientid', observed=True).subtype.first().to_dict()
EMT_SIGS = ['pEMT_puram', 'EMT_hallmark', 'EMT_tan']

print(f'\nPatients ({len(PATIENTS)}): ' + ', '.join(f'{p}[{SUBTYPE[p]}]' for p in PATIENTS))

print('\n--- SPOTS PER PATIENT BY REGION ---')
tab = pd.crosstab(df.patientid, df.tumor_region.fillna('non-tumor'))
print(tab.to_string())

print('\n--- SPOTS WITH STEMNESS, BY REGION (this limits the front/core test) ---')
ds = df[df.spatial_stemness.notna()]
tab_s = pd.crosstab(ds.patientid, ds.tumor_region.fillna('non-tumor'))
print(tab_s.to_string())

MIN_SPOTS = 10
testable = [p for p in PATIENTS
            if tab_s.loc[p].get('front', 0) >= MIN_SPOTS
            and tab_s.loc[p].get('core', 0) >= MIN_SPOTS]
print(f'\nPatients testable for front-vs-core stemness (>={MIN_SPOTS} spots each): '
      f'{len(testable)} -> {testable}')
print(f'Excluded: {[p for p in PATIENTS if p not in testable]}')

n = len(testable)
print(f'\nPOWER NOTE: with n={n} patients, the smallest attainable two-sided '
      f'Wilcoxon signed-rank p is {2/2**n:.3f}')
if 2/2**n > 0.05:
    print('  -> p<0.05 is MATHEMATICALLY IMPOSSIBLE for this test. '
          'Report effect sizes and direction, not significance.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==

In [2]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

VARS = ['spatial_stemness'] + EMT_SIGS
per_patient, summary = [], []

for var in VARS:
    sub = df[df[var].notna()]
    ds_, ps_, ns_ = [], [], []
    for p in PATIENTS:
        s = sub[(sub.patientid == p) & (sub.tumor_region.isin(['front', 'core']))]
        f = s[s.tumor_region == 'front'][var].values
        c = s[s.tumor_region == 'core'][var].values
        if len(f) < MIN_SPOTS or len(c) < MIN_SPOTS:
            per_patient.append({'variable': var, 'patient': p, 'subtype': SUBTYPE[p],
                                'n_front': len(f), 'n_core': len(c),
                                'mean_front': np.nan, 'mean_core': np.nan,
                                'cohens_d': np.nan, 'mwu_p': np.nan, 'testable': False})
            continue
        d_val = cohens_d(f, c)
        mwu_p = mannwhitneyu(f, c, alternative='two-sided')[1]
        per_patient.append({'variable': var, 'patient': p, 'subtype': SUBTYPE[p],
                            'n_front': len(f), 'n_core': len(c),
                            'mean_front': round(f.mean(), 4), 'mean_core': round(c.mean(), 4),
                            'cohens_d': round(d_val, 3), 'mwu_p': mwu_p, 'testable': True})
        ds_.append(d_val); ps_.append(mwu_p); ns_.append(p)

    if len(ds_) >= 3:
        try:
            wp = wilcoxon(ds_)[1]
        except Exception:
            wp = np.nan
        floor = 2 / 2 ** len(ds_)
        summary.append({'variable': var, 'n_patients': len(ds_),
                        'front_higher': f'{sum(x > 0 for x in ds_)}/{len(ds_)}',
                        'median_d': round(np.median(ds_), 3),
                        'min_d': round(min(ds_), 3), 'max_d': round(max(ds_), 3),
                        'wilcoxon_p': round(wp, 4),
                        'p_floor': round(floor, 4),
                        'interpretable': floor <= 0.05})

pp = pd.DataFrame(per_patient)
sm = pd.DataFrame(summary)

print('--- PER-PATIENT front vs core ---')
for var in VARS:
    print(f'\n{var}:')
    v = pp[pp.variable == var].drop(columns='variable')
    print(v.to_string(index=False))

print('\n\n--- PATIENT-LEVEL SUMMARY ---')
print(sm.to_string(index=False))
print('\n  median_d: Cohen\'s d across patients (positive = front higher)')
print('  p_floor : smallest attainable Wilcoxon p at this n')
print('  interpretable: False means significance testing is uninformative here')

pp.to_csv(f'{RESULTS}/frontcore_per_patient.csv', index=False)
sm.to_csv(f'{RESULTS}/frontcore_summary.csv', index=False)
print(f'\nSaved -> {RESULTS}/frontcore_per_patient.csv, frontcore_summary.csv')

--- PER-PATIENT front vs core ---

spatial_stemness:
 patient subtype  n_front  n_core  mean_front  mean_core  cohens_d        mwu_p  testable
1142243F    TNBC      880    1604      0.1874     0.1711     0.179 2.332162e-05      True
1160920F    TNBC      160    2155      0.1707     0.1480     0.292 6.635868e-03      True
 CID4290      ER       61    1707      0.1482     0.1789    -0.317 3.078642e-02      True
 CID4465    TNBC        0     255         NaN        NaN       NaN          NaN     False
CID44971    TNBC        3      95         NaN        NaN       NaN          NaN     False
 CID4535      ER      120     333      0.2071     0.1385     0.844 9.531998e-13      True

pEMT_puram:
 patient subtype  n_front  n_core  mean_front  mean_core  cohens_d        mwu_p  testable
1142243F    TNBC     1071    1903      0.3663     0.3609     0.063 5.617202e-01      True
1160920F    TNBC      318    2571      0.2456     0.2208     0.427 1.478717e-10      True
 CID4290      ER       71    1917 

In [3]:
from scipy.spatial import cKDTree

def hex_steps(dr, dc):
    dr, dc = np.abs(dr), np.abs(dc)
    return np.where(dc >= dr, (dr + dc) / 2.0, dr)

dist_rows = []
for p in PATIENTS:
    s = df[(df.patientid == p) & df.array_row.notna()]
    tum = s[s.region_class == 'tumor']
    non = s[s.region_class.isin(['non_tumor', 'in_situ'])]
    if len(non) == 0:
        print(f'{p}: no non-tumor reference spots, skipped')
        continue

    non_rc = non[['array_row', 'array_col']].values.astype(float)
    tree = cKDTree(non_rc)
    tum_rc = tum[['array_row', 'array_col']].values.astype(float)

    # take 30 Euclidean-nearest candidates, then pick the true hex minimum among them
    k = min(30, len(non_rc))
    _, idx = tree.query(tum_rc, k=k)
    idx = np.atleast_2d(idx.T).T if k > 1 else idx.reshape(-1, 1)

    dmin = []
    for i in range(len(tum_rc)):
        cand = non_rc[idx[i]]
        steps = hex_steps(cand[:, 0] - tum_rc[i, 0], cand[:, 1] - tum_rc[i, 1])
        dmin.append(steps.min())

    dist_rows.append(pd.DataFrame({'SpotID': tum.SpotID.values,
                                   'dist_to_nontumor': np.array(dmin)}))

dist = pd.concat(dist_rows, ignore_index=True)
df = df.merge(dist, on='SpotID', how='left')
print(f'Distance computed for {df.dist_to_nontumor.notna().sum()} tumor spots\n')

print('--- SANITY: distance vs the categorical front/core label ---')
chk = df[df.dist_to_nontumor.notna()].groupby('tumor_region', observed=True)['dist_to_nontumor']
print(chk.describe()[['count', 'mean', '50%', 'min', 'max']].round(2).to_string())
print('\n(front should be near 1, core clearly larger - confirms the distance is correct)')

print('\n--- DISTANCE RANGE PER PATIENT ---')
print(df[df.dist_to_nontumor.notna()].groupby('patientid', observed=True)['dist_to_nontumor']
      .agg(['count', 'median', 'max']).round(2).to_string())

print('\n\n--- SPEARMAN: score vs distance from tumor-nontumor interface ---')
print('    (negative = score decreases going deeper into tumor)\n')
rows = []
for var in VARS:
    rs, ns = [], []
    for p in PATIENTS:
        s = df[(df.patientid == p) & df.dist_to_nontumor.notna() & df[var].notna()]
        if len(s) < 50:
            continue
        rs.append(spearmanr(s[var], s.dist_to_nontumor)[0]); ns.append(p)
    try:
        wp = wilcoxon(rs)[1]
    except Exception:
        wp = np.nan
    rows.append({'variable': var, 'n_patients': len(rs),
                 'median_rho': round(np.median(rs), 3),
                 'negative': f'{sum(r < 0 for r in rs)}/{len(rs)}',
                 'wilcoxon_p': round(wp, 4),
                 **{p: round(r, 3) for p, r in zip(ns, rs)}})

distcorr = pd.DataFrame(rows)
print(distcorr.to_string(index=False))

distcorr.to_csv(f'{RESULTS}/distance_correlation.csv', index=False)
df[['SpotID', 'patientid', 'dist_to_nontumor']].dropna().to_csv(
    f'{RESULTS}/spot_distances.csv', index=False)
print(f'\nSaved -> {RESULTS}/distance_correlation.csv, spot_distances.csv')

Distance computed for 11300 tumor spots

--- SANITY: distance vs the categorical front/core label ---
               count   mean   50%  min   max
tumor_region                                
core          7783.0  16.70  12.0  1.0  62.0
front         1732.0   1.50   2.0  1.0   2.0
uncertain     1785.0  11.99   5.0  2.0  64.0

(front should be near 1, core clearly larger - confirms the distance is correct)

--- DISTANCE RANGE PER PATIENT ---
           count  median   max
patientid                     
1142243F    3627     4.0  18.0
1160920F    3146    13.0  64.0
CID4290     2297    25.0  62.0
CID4465     1131    18.0  44.0
CID44971     317     5.0  16.0
CID4535      782     6.0  28.0


--- SPEARMAN: score vs distance from tumor-nontumor interface ---
    (negative = score decreases going deeper into tumor)

        variable  n_patients  median_rho negative  wilcoxon_p  1142243F  1160920F  CID4290  CID4465  CID44971  CID4535
spatial_stemness           6      -0.113      5/6      0.0625 

In [4]:
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

m = df[df.dist_to_nontumor.notna()].copy()
zc = lambda s: (s - s.mean()) / s.std()
for var in VARS + ['dist_to_nontumor', 'Cancer Epithelial', 'CAFs', 'nCount_RNA']:
    m[f'z_{var}'] = m.groupby('patientid', observed=True)[var].transform(zc)
m['log_dist'] = np.log1p(m.dist_to_nontumor)
m['z_log_dist'] = m.groupby('patientid', observed=True)['log_dist'].transform(zc)

def fit_lmm(data, formula, re_formula, label):
    d0 = data.dropna(subset=[c for c in data.columns if c.startswith('z_')], how='any')
    try:
        fit = smf.mixedlm(formula, d0, groups=d0['patientid'],
                          re_formula=re_formula).fit(method='lbfgs')
        return {'model': label, 'n_obs': int(fit.nobs), 'converged': fit.converged,
                'terms': fit.fe_params.to_dict(), 'pvals': fit.pvalues.to_dict(),
                'ci': fit.conf_int().to_dict('index')}
    except Exception as e:
        return {'model': label, 'error': str(e)[:120]}

def show(res, focus):
    if 'error' in res:
        print(f'  {res["model"]}: FAILED - {res["error"]}'); return
    b  = res['terms'].get(focus, np.nan)
    p  = res['pvals'].get(focus, np.nan)
    ci = res['ci'].get(focus, {})
    lo, hi = ci.get(0, np.nan), ci.get(1, np.nan)
    print(f'  {res["model"]:<44} beta={b:+.4f}  95%CI[{lo:+.3f},{hi:+.3f}]  '
          f'p={p:.2e}  n={res["n_obs"]}  conv={res["converged"]}')

print('=== (a) DOES DISTANCE FROM THE INTERFACE PREDICT STEMNESS? ===')
ms = m[m.spatial_stemness.notna()].copy()
show(fit_lmm(ms, 'z_spatial_stemness ~ z_log_dist', '~z_log_dist',
             'stemness ~ distance'), 'z_log_dist')
show(fit_lmm(ms, 'z_spatial_stemness ~ z_log_dist + Q("z_Cancer Epithelial") + z_CAFs',
             '~z_log_dist', 'stemness ~ distance + composition'), 'z_log_dist')

print('\n=== (b) STEMNESS x EMT UNDER A MIXED MODEL ===')
for sig in EMT_SIGS:
    md = m[m.spatial_stemness.notna() & m[sig].notna()].copy()
    show(fit_lmm(md, f'z_spatial_stemness ~ z_{sig}', f'~z_{sig}',
                 f'stemness ~ {sig}'), f'z_{sig}')

print('\n=== (c) SAME, WITH COMPOSITION AND LIBRARY SIZE CONTROLLED ===')
for sig in EMT_SIGS:
    md = m[m.spatial_stemness.notna() & m[sig].notna()].copy()
    show(fit_lmm(md,
                 f'z_spatial_stemness ~ z_{sig} + Q("z_Cancer Epithelial") + z_CAFs + z_nCount_RNA',
                 f'~z_{sig}', f'stemness ~ {sig} + covariates'), f'z_{sig}')

print('\n=== (d) EMT SCORES vs DISTANCE ===')
for sig in EMT_SIGS:
    md = m[m[sig].notna()].copy()
    show(fit_lmm(md, f'z_{sig} ~ z_log_dist', '~z_log_dist',
                 f'{sig} ~ distance'), 'z_log_dist')

print('\n  beta = change in outcome (SD units) per 1 SD increase in predictor')
print('  random intercept AND random slope per patient in every model')

lmm_rows = []
for lbl, res in [('stemness~dist', fit_lmm(ms, 'z_spatial_stemness ~ z_log_dist', '~z_log_dist', 'a1'))]:
    pass
print('\n(Full model objects kept in memory; summary table written in step 2.6)')

=== (a) DOES DISTANCE FROM THE INTERFACE PREDICT STEMNESS? ===
  stemness ~ distance                          beta=-0.1394  95%CI[-0.269,-0.010]  p=3.50e-02  n=8583  conv=False
  stemness ~ distance + composition            beta=-0.1425  95%CI[-0.347,+0.062]  p=1.72e-01  n=8583  conv=True

=== (b) STEMNESS x EMT UNDER A MIXED MODEL ===
  stemness ~ pEMT_puram                        beta=+0.4451  95%CI[+0.268,+0.622]  p=7.97e-07  n=8583  conv=True
  stemness ~ EMT_hallmark                      beta=+0.4988  95%CI[+0.288,+0.710]  p=3.69e-06  n=8583  conv=True
  stemness ~ EMT_tan                           beta=+0.6495  95%CI[+0.432,+0.867]  p=4.68e-09  n=8583  conv=True

=== (c) SAME, WITH COMPOSITION AND LIBRARY SIZE CONTROLLED ===
  stemness ~ pEMT_puram + covariates           beta=+0.1728  95%CI[-0.004,+0.350]  p=5.53e-02  n=8583  conv=False
  stemness ~ EMT_hallmark + covariates         beta=+0.1713  95%CI[+0.023,+0.320]  p=2.38e-02  n=8583  conv=False
  stemness ~ EMT_tan + covariat

In [5]:
def fit_lmm2(data, formula, re_formula, label, method='lbfgs'):
    d0 = data.dropna(subset=[c for c in data.columns if c.startswith('z_')], how='any')
    try:
        fit = smf.mixedlm(formula, d0, groups=d0['patientid'],
                          re_formula=re_formula).fit(method=method)
        return fit
    except Exception:
        return None

def compare(data, formula, focus, label):
    out = []
    for re_f, re_lbl in [(f'~{focus}', 'rand. slope'), ('1', 'rand. intercept only')]:
        for meth in ['lbfgs', 'powell']:
            fit = fit_lmm2(data, formula, re_f, label, meth)
            if fit is None:
                continue
            out.append({'random_effects': re_lbl, 'optimizer': meth,
                        'beta': round(fit.fe_params.get(focus, np.nan), 4),
                        'p': f'{fit.pvalues.get(focus, np.nan):.2e}',
                        'converged': fit.converged})
    print(f'\n{label}')
    print(pd.DataFrame(out).to_string(index=False))

print('=== CONVERGENCE CHECK: does the estimate depend on model spec? ===')
ms = m[m.spatial_stemness.notna()].copy()
compare(ms, 'z_spatial_stemness ~ z_log_dist', 'z_log_dist', '(a) stemness ~ distance')
compare(ms, 'z_spatial_stemness ~ z_pEMT_puram', 'z_pEMT_puram', '(b) stemness ~ pEMT_puram')
compare(ms, 'z_spatial_stemness ~ z_pEMT_puram + Q("z_Cancer Epithelial") + z_CAFs + z_nCount_RNA',
        'z_pEMT_puram', '(c) stemness ~ pEMT_puram + covariates')
compare(ms, 'z_spatial_stemness ~ z_EMT_hallmark + Q("z_Cancer Epithelial") + z_CAFs + z_nCount_RNA',
        'z_EMT_hallmark', '(c) stemness ~ EMT_hallmark + covariates')
compare(ms, 'z_pEMT_puram ~ z_log_dist', 'z_log_dist', '(d) pEMT_puram ~ distance')

=== CONVERGENCE CHECK: does the estimate depend on model spec? ===

(a) stemness ~ distance
      random_effects optimizer    beta        p  converged
         rand. slope     lbfgs -0.1394 3.50e-02      False
         rand. slope    powell -0.1395 2.95e-04       True
rand. intercept only     lbfgs -0.1206 4.68e-27       True
rand. intercept only    powell -0.1196 6.50e-27       True

(b) stemness ~ pEMT_puram
      random_effects optimizer   beta         p  converged
         rand. slope     lbfgs 0.4451  7.97e-07       True
         rand. slope    powell 0.4448  1.78e-08       True
rand. intercept only     lbfgs 0.3304 1.94e-172       True
rand. intercept only    powell 0.3281 3.44e-169       True

(c) stemness ~ pEMT_puram + covariates
      random_effects optimizer   beta        p  converged
         rand. slope     lbfgs 0.1728 5.53e-02      False
         rand. slope    powell 0.1705 2.36e-02       True
rand. intercept only     lbfgs 0.0879 5.42e-09       True
rand. intercept onl

In [6]:
rng = np.random.default_rng(42)
N_PERM = 500

def toroidal_test(sub, var_a, var_b, n_perm=N_PERM):
    rr = sub.array_row.astype(int).values
    cc = sub.array_col.astype(int).values
    a  = sub[var_a].values
    b  = sub[var_b].values
    obs = spearmanr(a, b)[0]

    pos = {(r, c): i for i, (r, c) in enumerate(zip(rr, cc))}
    R0, R1 = rr.min(), rr.max()
    C0, C1 = cc.min(), cc.max()
    Rspan, Cspan = R1 - R0 + 1, C1 - C0 + 1

    null = []
    attempts = 0
    while len(null) < n_perm and attempts < n_perm * 10:
        attempts += 1
        dr = int(rng.integers(0, Rspan))
        dc = int(rng.integers(0, Cspan))
        if (dr + dc) % 2 != 0 or (dr, dc) == (0, 0):
            continue
        pairs = []
        for i in range(len(rr)):
            nr = R0 + (rr[i] - R0 + dr) % Rspan
            nc = C0 + (cc[i] - C0 + dc) % Cspan
            j = pos.get((nr, nc))
            if j is not None:
                pairs.append((i, j))
        if len(pairs) < 50:
            continue
        ii = np.array([p[0] for p in pairs])
        jj = np.array([p[1] for p in pairs])
        null.append(spearmanr(a[ii], b[jj])[0])

    null = np.array(null)
    p_emp = (np.sum(np.abs(null) >= abs(obs)) + 1) / (len(null) + 1)
    return obs, null, p_emp

print(f'Toroidal shift test: {N_PERM} permutations per patient per signature')
print('H0: no spatial association between the two fields,')
print('    while each field keeps its own autocorrelation structure\n')

perm_rows, null_store = [], {}
dp = df[df.spatial_stemness.notna() & df.array_row.notna()]

for sig in EMT_SIGS:
    print(f'--- {sig} ---')
    for p in PATIENTS:
        s = dp[(dp.patientid == p) & dp[sig].notna()]
        if len(s) < 100:
            print(f'  {p:10s} skipped (n={len(s)})')
            continue
        obs, null, pe = toroidal_test(s, 'spatial_stemness', sig)
        null_store[(sig, p)] = null
        perm_rows.append({'signature': sig, 'patient': p, 'subtype': SUBTYPE[p],
                          'n_spots': len(s), 'observed_rho': round(obs, 3),
                          'null_mean': round(null.mean(), 3),
                          'null_sd': round(null.std(), 3),
                          'n_valid_perm': len(null),
                          'p_spatial': round(pe, 4)})
        print(f'  {p:10s} n={len(s):5d}  obs={obs:+.3f}  '
              f'null={null.mean():+.3f}+/-{null.std():.3f}  '
              f'p={pe:.4f}  ({len(null)} perms)')
    print()

perm = pd.DataFrame(perm_rows)

print('--- COMBINED ACROSS PATIENTS (Fisher) ---')
from scipy.stats import combine_pvalues
for sig in EMT_SIGS:
    sub = perm[perm.signature == sig]
    fp = combine_pvalues(sub.p_spatial.values)[1]
    print(f'  {sig:14s} {sum(sub.p_spatial < 0.05)}/{len(sub)} patients p<0.05  |  '
          f'Fisher combined p={fp:.2e}')

perm.to_csv(f'{RESULTS}/spatial_permutation.csv', index=False)
np.savez(f'{RESULTS}/permutation_nulls.npz',
         **{f'{s}__{p}': v for (s, p), v in null_store.items()})
print(f'\nSaved -> {RESULTS}/spatial_permutation.csv')
print(f'Saved -> {RESULTS}/permutation_nulls.npz  (null distributions for Fig 4c)')

Toroidal shift test: 500 permutations per patient per signature
H0: no spatial association between the two fields,
    while each field keeps its own autocorrelation structure

--- pEMT_puram ---
  1142243F   n= 3361  obs=+0.125  null=+0.000+/-0.029  p=0.0020  (500 perms)
  1160920F   n= 3103  obs=+0.161  null=-0.005+/-0.052  p=0.0020  (500 perms)
  CID4290    n= 2048  obs=+0.377  null=-0.003+/-0.054  p=0.0020  (500 perms)
  CID4465    n=  343  obs=+0.161  null=+0.005+/-0.146  p=0.2827  (328 perms)
  CID44971   n=  704  obs=+0.231  null=-0.012+/-0.122  p=0.0818  (500 perms)
  CID4535    n=  704  obs=+0.534  null=-0.054+/-0.184  p=0.0020  (500 perms)

--- EMT_hallmark ---
  1142243F   n= 3361  obs=+0.166  null=-0.004+/-0.038  p=0.0020  (500 perms)
  1160920F   n= 3103  obs=+0.161  null=-0.011+/-0.063  p=0.0020  (500 perms)
  CID4290    n= 2048  obs=+0.469  null=-0.003+/-0.053  p=0.0020  (500 perms)
  CID4465    n=  343  obs=+0.110  null=+0.004+/-0.144  p=0.4303  (336 perms)
  CID44971  

In [7]:
from statsmodels.stats.multitest import multipletests

fam = []

for _, r in sm.iterrows():
    fam.append({'family': 'front_vs_core', 'test': f'front vs core: {r.variable}',
                'statistic': f"median d={r.median_d}", 'n': r.n_patients,
                'p_raw': r.wilcoxon_p,
                'note': 'power-limited (p floor 0.125)' if not r.interpretable else ''})

for _, r in distcorr.iterrows():
    fam.append({'family': 'distance', 'test': f'distance: {r.variable}',
                'statistic': f"median rho={r.median_rho}", 'n': r.n_patients,
                'p_raw': r.wilcoxon_p, 'note': ''})

for sig in EMT_SIGS:
    sub = perm[perm.signature == sig]
    fam.append({'family': 'spatial_permutation', 'test': f'spatial assoc: {sig}',
                'statistic': f"{sum(sub.p_spatial<0.05)}/{len(sub)} patients",
                'n': len(sub),
                'p_raw': combine_pvalues(sub.p_spatial.values)[1], 'note': ''})

fam = pd.DataFrame(fam)
rej, p_adj, _, _ = multipletests(fam.p_raw.values, alpha=0.05, method='fdr_bh')
fam['p_fdr'] = p_adj
fam['significant'] = rej
fam['p_raw'] = fam.p_raw.map(lambda x: f'{x:.2e}')
fam['p_fdr'] = fam.p_fdr.map(lambda x: f'{x:.2e}')

print(f'=== PRIMARY HYPOTHESIS FAMILY: {len(fam)} tests, Benjamini-Hochberg FDR ===\n')
for f in ['front_vs_core', 'distance', 'spatial_permutation']:
    print(f'--- {f} ---')
    print(fam[fam.family == f].drop(columns='family').to_string(index=False))
    print()

print(f'Significant after FDR: {fam.significant.sum()}/{len(fam)}')
print('\nNOTE: validation layers, partial correlations and sensitivity analyses')
print('      are NOT in this family; they are reported as exploratory, uncorrected.')

fam.to_csv(f'{RESULTS}/statistical_tables.csv', index=False)
print(f'\nSaved -> {RESULTS}/statistical_tables.csv')

print('\n' + '=' * 70)
print('STEP 2 SUMMARY')
print('=' * 70)
print(f'  spots analysed          : {len(df)} total, {df.spatial_stemness.notna().sum()} with stemness')
print(f'  patients                : {len(PATIENTS)} ({sum(v=="TNBC" for v in SUBTYPE.values())} TNBC, '
      f'{sum(v=="ER" for v in SUBTYPE.values())} ER+)')
print(f'  front/core testable     : {len(testable)} patients for stemness, 6 for EMT')
print(f'  distance computed       : {df.dist_to_nontumor.notna().sum()} tumor spots')
print(f'  permutations run        : {len(perm)} tests x {N_PERM} shifts')
print(f'  primary family          : {len(fam)} tests, {fam.significant.sum()} significant after FDR')
print('\n  output files in 04_results/:')
for f in ['frontcore_per_patient.csv', 'frontcore_summary.csv', 'distance_correlation.csv',
          'spot_distances.csv', 'spatial_permutation.csv', 'permutation_nulls.npz',
          'statistical_tables.csv']:
    print(f'    - {f}')

=== PRIMARY HYPOTHESIS FAMILY: 11 tests, Benjamini-Hochberg FDR ===

--- front_vs_core ---
                           test      statistic  n    p_raw                          note    p_fdr  significant
front vs core: spatial_stemness median d=0.236  4 6.25e-01 power-limited (p floor 0.125) 6.25e-01        False
      front vs core: pEMT_puram median d=0.161  6 5.62e-01                               6.19e-01        False
    front vs core: EMT_hallmark median d=0.403  6 1.56e-01                               2.45e-01        False
         front vs core: EMT_tan median d=1.006  6 3.12e-02                               6.86e-02        False

--- distance ---
                      test         statistic  n    p_raw note    p_fdr  significant
distance: spatial_stemness median rho=-0.113  6 6.25e-02      1.15e-01        False
      distance: pEMT_puram  median rho=-0.06  6 4.38e-01      5.35e-01        False
    distance: EMT_hallmark median rho=-0.085  6 3.12e-01      4.30e-01        False
